# Cuadernillo de preparacion del dataset Adult (UCI)

Este cuadernillo une y prepara `adult.data` (entrenamiento) y `adult.test` (prueba) para ejercicios de:

- Clasificacion (objetivo principal: ingreso `>50K` vs `<=50K`)
- Regresion (objetivo sugerido: `hours-per-week`)
- Redes neuronales (entrada numerica densa)

## Contexto general

El dataset **Adult Census Income** contiene atributos demograficos y laborales de personas extraidos del censo de EE. UU. El objetivo clasico es predecir si una persona gana mas de 50K USD al ano.

- `adult.data`: conjunto de entrenamiento original.
- `adult.test`: conjunto de prueba original.

Particularidades importantes de `adult.test`:
1. La primera linea es un encabezado tecnico (`|1x3 Cross validator`) y no es un registro.
2. Las etiquetas de clase vienen con punto final (`<=50K.` y `>50K.`), por lo que deben normalizarse para que coincidan con entrenamiento.


## Diccionario de columnas (aplica para ambos: `adult.data` y `adult.test`)

| Columna | Tipo original | Significado |
|---|---|---|
| `age` | Continua | Edad de la persona. |
| `workclass` | Categórica | Tipo de empleo (privado, gobierno, autoempleo, etc.). |
| `fnlwgt` | Continua | Peso muestral final del censo. |
| `education` | Categórica | Nivel educativo textual. |
| `education-num` | Continua/entera | Nivel educativo numerico (ordinal). |
| `marital-status` | Categórica | Estado civil. |
| `occupation` | Categórica | Ocupacion principal. |
| `relationship` | Categórica | Relacion familiar en el hogar. |
| `race` | Categórica | Grupo racial reportado. |
| `sex` | Categórica | Sexo biologico reportado. |
| `capital-gain` | Continua | Ganancia de capital. |
| `capital-loss` | Continua | Perdida de capital. |
| `hours-per-week` | Continua | Horas trabajadas por semana. |
| `native-country` | Categórica | Pais de origen. |
| `income` | Categórica objetivo | Clase objetivo: `<=50K` o `>50K`. |

**Nota:** ambos datasets comparten el mismo esquema de columnas.


## Carga, limpieza y estandarizacion

Transformaciones que haremos y por que:

1. **Eliminar cabecera tecnica de `adult.test`**: no es un dato real.
2. **Quitar espacios extra**: en este dataset hay espacios despues de comas; limpiarlos evita categorias duplicadas.
3. **Normalizar etiqueta `income` en test**: eliminar `.` final para igualar formatos.
4. **Tratar `?` como faltante**: representa valor desconocido; se convierte a `NaN` para imputacion consistente.
5. **Convertir objetivo a binario**: `<=50K -> 0`, `>50K -> 1` para modelos de ML.
6. **Codificar categoricas con One-Hot**: necesario para modelos numericos y evita asumir orden artificial en categorias nominales.
7. **Estandarizar numericas en pipelines sensibles a escala** (regresion y red neuronal): mejora estabilidad y convergencia.


In [ ]:
import pandas as pd
import numpy as np
import torch 
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

base_path = Path('.')

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

adult_train = pd.read_csv(base_path / 'adult.data', header=None, names=columns, skipinitialspace=True)

adult_test = pd.read_csv(
    base_path / 'adult.test',
    header=None,
    names=columns,
    skipinitialspace=True,
    skiprows=1
)

# Limpieza de etiquetas y espacios
adult_train['income'] = adult_train['income'].astype(str).str.strip()
adult_test['income'] = adult_test['income'].astype(str).str.strip().str.replace('.', '', regex=False)

for c in columns:
    if adult_train[c].dtype == 'object':
        adult_train[c] = adult_train[c].astype(str).str.strip()
    if adult_test[c].dtype == 'object':
        adult_test[c] = adult_test[c].astype(str).str.strip()

# Reemplazar '?' por NaN en categoricas
for df in [adult_train, adult_test]:
    df.replace('?', np.nan, inplace=True)

# Objetivo binario
target_map = {'<=50K': 0, '>50K': 1}
adult_train['income_bin'] = adult_train['income'].map(target_map)
adult_test['income_bin'] = adult_test['income'].map(target_map)

print('Train shape:', adult_train.shape)
print('Test shape :', adult_test.shape)
print('\nDistribucion train income_bin:')
print(adult_train['income_bin'].value_counts(dropna=False))
print('\nDistribucion test income_bin:')
print(adult_test['income_bin'].value_counts(dropna=False))


ImportError: DLL load failed while importing lib: No se encontró el proceso especificado.

In [ ]:
# Definicion de tipos de variables
numeric_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

X_train = adult_train[numeric_features + categorical_features].copy()
y_train_clf = adult_train['income_bin'].copy()

X_test = adult_test[numeric_features + categorical_features].copy()
y_test_clf = adult_test['income_bin'].copy()

print('X_train:', X_train.shape, '| y_train_clf:', y_train_clf.shape)
print('X_test :', X_test.shape, '| y_test_clf :', y_test_clf.shape)


## Preparacion para REGRESION (ejercicio sugerido)

Como el objetivo principal del dataset es clasificacion, para practicar regresion proponemos predecir `hours-per-week`.

- Variable objetivo regresion: `hours-per-week`.
- Se excluye `hours-per-week` de los predictores para evitar fuga de informacion.


In [ ]:
reg_target = 'hours-per-week'

reg_numeric_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss']
reg_categorical_features = categorical_features.copy()

X_train_reg = adult_train[reg_numeric_features + reg_categorical_features].copy()
y_train_reg = adult_train[reg_target].copy()

X_test_reg = adult_test[reg_numeric_features + reg_categorical_features].copy()
y_test_reg = adult_test[reg_target].copy()

reg_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), reg_numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), reg_categorical_features)
    ]
)

X_train_reg_ready = reg_preprocessor.fit_transform(X_train_reg)
X_test_reg_ready = reg_preprocessor.transform(X_test_reg)

print('Regresion listo:')
print('X_train_reg_ready:', X_train_reg_ready.shape, '| y_train_reg:', y_train_reg.shape)
print('X_test_reg_ready :', X_test_reg_ready.shape,  '| y_test_reg :', y_test_reg.shape)


NameError: name 'categorical_features' is not defined

In [ ]:
m_train =  y_train_reg.size
m_test = y_test_reg.size

# NORMALIZACION

In [ ]:
def featureNormalize(X):
    X_norm = X.copy()
    mu = np.zeros(X.shape[1])
    sigma = np.zeros(X.shape[1])
    
    mu = np.mean(X, axis = 0)
    sigma = np.std(X, axis = 0)
    X_norm = (X - mu) / sigma
    
    return X_norm, mu, sigma

In [ ]:
X_train_reg_norm, mu, sigma = featureNormalize(X_train_reg)
print('Media calculada:', mu)
print('Desviación estandar calculada:', sigma)
print(X_train_reg_norm)

X_test_reg_norm, mu, sigma = featureNormalize(X_test_reg)
print('Media calculada:', mu)
print('Desviación estandar calculada:', sigma)
print(X_test_reg_norm)

Despues de featureNormalize la funcion es provada, se añade el temino de interseccion a X_norm:

In [ ]:
X_trainR = np.concatenate([np.ones((m_train, 1)), X_train_reg_norm], axis=1)

In [ ]:
X_testR = np.concatenate([np.ones((m_test, 1)), X_test_reg_norm], axis=1)

Definimos la funcion de costo y Desenso por el gradiente

In [ ]:
def computeCostMulti(X, y, theta):
    # Inicializa algunos valores utiles
    m = y.shape[0] # numero de ejemplos de entrenamiento

    J = 0

    # h = np.dot(X, theta)

    J = (1/(2 * m)) * np.sum(np.square(np.dot(X, theta) - y))

    return J

In [ ]:
def gradientDescentMulti(X, y, theta, alpha, num_iters):

    # Inicializa algunos valores
    m = y.shape[0] # numero de ejemplos de entrenamiento

    # realiza una copia de theta, el cual será acutalizada por el descenso por el gradiente

    theta = theta.copy()

    J_history = []

    for i in range(num_iters):
        theta = theta - (alpha / m) * (np.dot(X, theta) - y).dot(X)
        J_history.append(computeCostMulti(X, y, theta))

    return theta, J_history

In [ ]:
# Librerias para graficación (trazado de gráficos)
from matplotlib import pyplot
from mpl_toolkits.mplot3d import Axes3D  # Necesario para graficar superficies 3D
# Elegir algun valor para alpha (probar varias alternativas)
alpha = 0.001 # alpha = 0.003
num_iters = 10000

# inicializa theta y ejecuta el descenso por el gradiente
theta = np.zeros(3)
theta, J_history = gradientDescentMulti(X_trainR, y_train_reg, theta, alpha, num_iters)

# Grafica la convergencia del costo
pyplot.plot(np.arange(len(J_history)), J_history, lw=2)
pyplot.xlabel('Numero de iteraciones')
pyplot.ylabel('Costo J')

# Muestra los resultados del descenso por el gradiente
print('theta calculado por el descenso por el gradiente: {:s}'.format(str(theta)))

## PRUEBA CON DATOS DE TEST

In [ ]:
HTPS = price = np.dot(X_testR[2], theta) 
print('Las horas trabajadas para el tercer ejemplo de teste es (usando el descenso por el gradiente): ${:.0f}'.format(HTPS))

NameError: name 'X_testR' is not defined

# PRESICION

## Resumen de cambios realizados

- Se quitaron inconsistencias de formato (`espacios`, `.` final en clase de test).
- Se trataron valores desconocidos (`?`) como faltantes para imputacion formal.
- Se codifico el objetivo de ingreso a binario (`0/1`).
- Se aplico One-Hot a categoricas para uso en modelos numericos.
- Se estandarizaron variables numericas para mejorar entrenamiento en modelos sensibles a escala.

Con esto ya tienes conjuntos listos:

- Clasificacion: `X_train_clf_ready`, `y_train_clf`, `X_test_clf_ready`, `y_test_clf`
- Regresion: `X_train_reg_ready`, `y_train_reg`, `X_test_reg_ready`, `y_test_reg`
- Red neuronal: `X_train_nn`, `y_train_nn`, `X_test_nn`, `y_test_nn`
